In [1]:
import requests 
from bs4 import BeautifulSoup as bs
import pandas as pd
import tldextract
import numpy as np
import matplotlib.pylab as plt
import os
import re

In [2]:
def on_sale_chk(text):
    if len(text)<1:
        return False
    return 'domain' in text and 'sale' in text

def on_parked_chk(text):
    if len(text)<1:
        return True
    return 'domain' in text and 'park' in text

def on_Parked(text):
    if len(text)<1:
        return True
    return (('website' in text or 'content' in text) and 'unavailable' in text) or ('will' in text and 'soon' in text)

In [6]:
#returns html contents, textual character length, website size, status code, parked or on sale

def soupFromUrl(scrapeUrl):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
    try:
        req = requests.get(scrapeUrl, headers=headers, timeout=5)
        # print(req.status_code)
        req.close()
        if req.status_code == 200:
            # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
            soup = bs(req.text,'html.parser')

            # print(soup)

            text = ''

            if soup.body:
                text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

            # print(soup)

            # print('text',text)
            # print(bs(req.text, 'html.parser'))
            # return [bs(req.text, 'html.parser'),len(req.text), len(req.content), req.status_code]
            # print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])
            return [len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))]
        else:
            # return [-1,0,0,req.status_code]
            return [0,0,req.status_code,0]
    except:
        # return [-1,0,0,-1]
        return [0,0,-1,0]

In [5]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
req = requests.get('https://www.lycos.com/', headers=headers, timeout=5)
print(req.status_code)
req.close()
if req.status_code == 200:
    # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
    soup = bs(req.text,'html.parser')

    # print(soup)
    text = ''

    if soup.body:
        text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

    print(soup)
    print('text',text)
    print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])

200
<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<!-- The above 3 meta tags *must* come first in the head; any other head content must come *after* these tags -->
<meta content="Lycos, Inc., is a web search engine and web portal established in 1994, spun out of Carnegie Mellon University. Lycos also encompasses a network of email, webhosting, social networking, and entertainment websites." name="description"/>
<meta content="" name="author"/>
<link href="https://ly.lygo.net/static/lycos/img/favicon.ico" rel="icon" type="image/png"/>
<title>Lycos.com</title>
<link href="//fonts.googleapis.com/css?family=Lato:400,300,300italic,400italic,700,700italic" rel="stylesheet" type="text/css"/>
<link href="/css/in/fonts.css" rel="stylesheet" type="text/css">
<link href="https://ly.lygo.net/static/lycos/css/in/font-awesome.css" rel="stylesheet" type="text

In [200]:
# print(soupFromUrl('https://www.delinian.com/'))
# print(soupFromUrl('https://www.makecashonline.com/'))
print(soupFromUrl('https://www.lycos.com/'))

[13609, 328, 200, 0]


In [7]:
enron_url = list(pd.read_csv('../Dataset/URL Data/enron Dataset.csv',delimiter='\t')['URL'])
enron_url[:10]

['http://www.hotmail.com',
 'http://www.hotmail.com',
 'http://www.hotmail.com-attl.htmmessage-id:from',
 'http://dl.www.juno.com/get/tagj',
 'http://www.juno.com/upgrade.ifyouareunable',
 'http://dl.www.juno.com/get/tagj',
 'http://www.juno.com/junofree',
 'http://dl.www.juno.com/get/tagj',
 'http://www.hotmail.com',
 'http://cdnow.com/myorder/otid=16840862youcanalso']

In [ ]:
import re

def find_first_slash_preceded_by_number(s):
    # Regular expression to find the first instance of a number followed by '/'
    match = re.search(r'\d+/', s)
    
    if match:
        return match.start() + len(match.group()) - 1  # Return the index of '/'
    else:
        return -1  # Return -1 if no match is found

def enron_refine(i):
    priority_tld = ['com','edu','org','net','biz','info','co.uk','co.nz','gov']
    other_tld = ['ru','su','uk','us','ws','ie']
    first = -1
    first = str(i).find('http')
    if first==-1:
        first = str(i).find('www.')
    if first==-1:
        continue
    last = -1
    for j in priority_tld:
        if '.'+str(j) in str(i):
            last = str(i).find(str(j))+len(j)
            break
    if last!=-1:
        return i[first:last]
    else:
        xxx = find_first_slash_preceded_by_number(i)
        if xxx!=-1:
            return i[first:xxx]
    return i

# Example usage
string = "example77/test 88/test2 99/test3"
index = find_first_slash_preceded_by_number(string)
print(index)  # Outputs the index of the first '/' preceded by a number

9


In [9]:
unique_enron_url = set()

priority_tld = ['com','edu','org','net','biz','info','co.uk','co.nz','gov']
other_tld = ['ru','su','uk','us','ws','ie']

In [10]:
for idx,i in enumerate(enron_url):
    first = -1
    first = str(i).find('http')
    if first==-1:
        first = str(i).find('www.')
    if first==-1:
        continue
    last = -1
    for j in priority_tld:
        if '.'+str(j) in str(i):
            last = str(i).find(str(j))+len(j)
            break
    if last!=-1:
        unique_enron_url.add(i[first:last])
    else:
        xxx = find_first_slash_preceded_by_number(i)
        if xxx!=-1:
            unique_enron_url.add(i[first:xxx])

if '' in unique_enron_url:
    unique_enron_url.remove('')
unique_enron_url = list(unique_enron_url)
len(unique_enron_url)

3460

In [11]:
for idx,i in enumerate(unique_enron_url):
    if '..' in i:
        if 'www' in i:
            unique_enron_url[idx] = ''
        else:
            unique_enron_url[idx] = unique_enron_url[idx].replace('..','.')

In [12]:
unique_enron_url = [i for i in unique_enron_url if i!='']

In [13]:
enron_dataset = {'ham':list(pd.read_csv('../Dataset/URL Data/enron Dataset Ham.csv',delimiter='\t')['URL']),'spam':list(pd.read_csv('../Dataset/URL Data/enron Dataset Spam.csv',delimiter='\t')['URL'])}

In [14]:
enron_url_in_ham = [0]*len(unique_enron_url)
enron_url_in_spam = [0]*len(unique_enron_url)

for idx,i in enumerate(unique_enron_url):
    for j in enron_dataset['ham']:
        if not isinstance(j, str):
            continue
        if i in j:
            enron_url_in_ham[idx] = 1
            break
    for j in enron_dataset['spam']:
        if not isinstance(j, str):
            continue
        if i in j:
            enron_url_in_spam[idx] = 1
            break

print(enron_url_in_ham.count(1))
print(enron_url_in_spam.count(1))

438
2970


In [15]:
common_urls = []
for i in range(len(enron_url_in_ham)):
    if enron_url_in_ham[i]==enron_url_in_spam[i]:
        common_urls.append(unique_enron_url[i])

len(common_urls)

76

In [16]:
enron_dataset['Unique Url'] = unique_enron_url

In [17]:
import tldextract

def FQDN(Url):
    
    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [18]:
enron_dataset['FQDN'] = [FQDN(i) for i in enron_dataset['Unique Url']]

len(set(enron_dataset['FQDN']))

3433

In [19]:
a = soupFromUrl('https://facebook.com')

print(a)

[73358, 645, 200, 0]


In [52]:
# enron_dataset = pd.read_csv('../Dataset/URL Data/Enron Websites Analysis.csv')

The below query last ran on 9 Jan 2025

In [20]:
website_size, text_content_length, status_code, parked = [],[],[],[]

for i in enron_dataset['FQDN']:
    a = soupFromUrl('https://'+i)

    website_size.append(a[0])
    text_content_length.append(a[1])
    status_code.append(a[2])
    parked.append(a[3])


# parked = [0]*len(enron_dataset)

# for idx,i in enumerate(enron_dataset['FQDN']):
#     if enron_dataset['Status Code'][idx]==200:
#         a = soupFromUrl('https://'+i)
#         parked[idx] = a[3]
#         # break

# print(parked)

In [21]:
enron_dataset['Website Size in KB'] = website_size
enron_dataset['Website Textual Content Length'] = text_content_length
enron_dataset['Status Code'] = status_code

enron_dataset['Parked'] = parked

In [22]:
for i in enron_dataset:
    print(len(enron_dataset[i]))

2389
8942
3456
3456
3456
3456
3456
3456


In [23]:
enron_dataset['ham'] = enron_url_in_ham
enron_dataset['spam'] = enron_url_in_spam

In [24]:
enron_dataset

{'ham': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  1,
  0,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0

In [25]:
# enron_dataset = pd.read_csv('../Dataset/URL Data/Enron Websites Analysis.csv')
enron_dataset = pd.DataFrame.from_dict(enron_dataset)
enron_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,1,http://wiseschool.com,wiseschool.com,0,0,-1,0
1,0,1,http://endosonic.com,endosonic.com,0,0,-1,0
2,0,1,http://otq.hellimnone.com,otq.hellimnone.com,0,0,-1,0
3,0,1,http://www.netcollage.com,www.netcollage.com,0,0,-1,0
4,0,1,http://bizarre.mainoemstore.com,bizarre.mainoemstore.com,0,0,-1,0


In [26]:
enron_dataset['Status Code'].value_counts()

Status Code
-1      2852
 200     562
 403      22
 404      14
 429       2
 401       1
 503       1
 500       1
 400       1
Name: count, dtype: int64

In [27]:
numbers_to_replace = [501,403, 401]

# Value to replace with
new_value = 200

# Update the column
enron_dataset.loc[enron_dataset['Status Code'].isin(numbers_to_replace), 'Status Code'] = new_value

In [28]:
enron_dataset['Status Code'].value_counts()

Status Code
-1      2852
 200     585
 404      14
 429       2
 503       1
 500       1
 400       1
Name: count, dtype: int64

In [29]:
print(len(enron_dataset[(enron_dataset['Status Code']==200) & (enron_dataset['ham']==1)]))
print(len(enron_dataset[(enron_dataset['Status Code']==200) & (enron_dataset['spam']==1)]))

198
366


In [30]:
enron_dataset['Parked'].value_counts()

Parked
0    3254
1     202
Name: count, dtype: int64

In [31]:
print(len(enron_dataset[(enron_dataset['Parked']==1) & (enron_dataset['ham']==1)]))
print(len(enron_dataset[(enron_dataset['Parked']==1) & (enron_dataset['ham']==0)]))

30
172


In [32]:
enron_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,1,http://wiseschool.com,wiseschool.com,0,0,-1,0
1,0,1,http://endosonic.com,endosonic.com,0,0,-1,0
2,0,1,http://otq.hellimnone.com,otq.hellimnone.com,0,0,-1,0
3,0,1,http://www.netcollage.com,www.netcollage.com,0,0,-1,0
4,0,1,http://bizarre.mainoemstore.com,bizarre.mainoemstore.com,0,0,-1,0


In [33]:
enron_dataset.to_csv('../Dataset/URL Data/Enron Websites Analysis.csv', index=None)